# 01 - Initial Scraping Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini digunakan untuk pengumpulan data awal dari **Source Accounts** (60 hari terakhir) dan **Search Keywords** (Core: 30 hari, Discovery: 30 hari).
- Mendukung seleksi ulang sumber secara manual (centang akun/keyword yang ingin dijalankan).
- Menyimpan raw post ke System Log sebelum pemrosesan AI.
- Melakukan deduplikasi otomatis berdasarkan Post ID (shortcode).
- Menetapkan `global_checkpoint` (waktu MULAI run) saat cakupan scraping run ini lengkap, sesuai D18: kegagalan ekstraksi Gemini tidak menahan checkpoint, tetapi run harus menyelesaikan minimal satu sumber; bila satu-satunya sumber tertunda kuota 100 post atau mentok ceiling, checkpoint tidak maju dan sumber itu dijalankan pertama pada run berikutnya.


### 0. Environment Check & Smart Install (wajib dijalankan pertama)
Sel pertama mencetak versi Python dan paket, lalu memeriksa import serta signature API yang dipakai `compmon`.
Sel kedua hanya memasang paket yang **belum ada atau terlalu lama** (tidak pernah `pip install -r`, tidak menurunkan paket bawaan Colab).
Jika ada paket yang dipasang: **Runtime > Restart session**, lalu jalankan ulang notebook dari atas.


In [ ]:
# @title 0a. Environment Check & Clone GitHub (jika di Colab)
# Memastikan repositori ter-clone di Colab, lalu memeriksa Python, versi paket, import, dan signature API.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/fajrialfarizy/SPIKE-Competition-Searching.git"
REPO_NAME = "SPIKE-Competition-Searching"

# Jika dijalankan di Google Colab (/content) dan compmon belum ada, clone dari GitHub
if not (Path.cwd() / 'compmon').exists() and not (Path.cwd().parent / 'compmon').exists():
    if not (Path.cwd() / REPO_NAME).exists():
        print(f"Mengunduh repositori dari GitHub: {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    if (Path.cwd() / REPO_NAME).exists():
        os.chdir(Path.cwd() / REPO_NAME)
        print(f"Beralih ke direktori proyek: {os.getcwd()}")

_root = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from compmon.env_check import run_environment_check

run_environment_check()


In [ ]:
# @title 0b. Smart Install
# Hanya memasang paket yang belum ada / di bawah versi minimum di requirements.txt.
# Tidak memakai `pip install -r` dan tidak menurunkan paket bawaan Colab.
from compmon.env_check import ensure_environment

ensure_environment()


### 1. Setup Lingkungan & Autentikasi Google Sheets
Memuat kode modul `compmon` (dependensi sudah dicek dan dipasang oleh sel 0) dan melakukan autentikasi via `google.auth.default()` + `gspread.authorize()`.

In [ ]:
# @title 1. Muat Kode
# Petunjuk: Pastikan folder 'compmon' dan 'requirements.txt' sudah diunggah ke runtime Colab,
# atau clone repositori jika menggunakan git.
import os, sys
from pathlib import Path

# Pastikan root project dan compmon masuk ke sys.path
root_dir = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from google.colab import auth, userdata
import google.auth
import gspread

# Autentikasi akun Google resmi Colab (Instruction 13)
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Membaca kredensial dari Colab Secrets
SPREADSHEET_ID = userdata.get('SPREADSHEET_ID')
APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### 2. Parameter Eksekusi (Colab Form)
Pilih sumber yang ingin dijalankan. Biarkan kosong untuk menjalankan semua akun & keyword aktif.

In [ ]:
# @title Konfigurasi Seleksi Sumber (Opsional)
# @markdown Kosongkan untuk menjalankan semua sumber akun dan keyword aktif.
selected_accounts_str = "" # @param {type:"string"}
selected_keywords_str = "" # @param {type:"string"}

selected_accounts = [a.strip() for a in selected_accounts_str.split(',') if a.strip()] or None
selected_keywords = [k.strip() for k in selected_keywords_str.split(',') if k.strip()] or None

### 3. Eksekusi Initial Scraping & AI Processing

In [ ]:
from compmon.sheets_repo import SheetsRepository
from compmon.pipeline import CompetitionPipeline

repo = SheetsRepository(gc, SPREADSHEET_ID)
pipeline = CompetitionPipeline(
    sheets_repo=repo,
    apify_token=APIFY_API_TOKEN,
    gemini_api_key=GEMINI_API_KEY,
    notebook_name="INITIAL"
)

summary = pipeline.run_initial_scraping(
    selected_accounts=selected_accounts,
    selected_keywords=selected_keywords
)

print("=== Ringkasan Eksekusi Initial Scraping ===")
for key, val in summary.items():
    print(f"{key}: {val}")